Check **Setup**

In [1]:
# platform - built-in python library
# gives info about the system we are running on
import platform
import subprocess
import os

In [2]:
print("Python version", platform.python_version())

Python version 3.12.13


In [3]:
import torch

if torch.cuda.is_available():
  print("GPU available")
else:
  print("No GPU - using CPU")

GPU available


Mount **Google Drive**

In [4]:
# drive lets us connect our Google Drive to this collab
from google.colab import drive
import os

# os helps to interact with the file system


drive.mount("/content/drive", force_remount=True)

PROJECT_DIR = "/content/drive/MyDrive/neurosynth"

os.makedirs(PROJECT_DIR, exist_ok=True)

# create a raw data folder - download EEG files go here
os.makedirs(f"{PROJECT_DIR}/data/raw", exist_ok=True)

# create the processed data folder - cleaned numpy arrays go here
os.makedirs(f"{PROJECT_DIR}/data/processed", exist_ok=True)

Mounted at /content/drive


Installing packages

In [5]:
# mne is a EEG processing library
# mlflow tracks our training experiments (loss, accuracy, parameters)

!pip install mne torch transformers mlflow scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93

In [6]:
import mne  # EEG processing
import torch  # deep learning
import transformers # transformer model building blocks
import sklearn  # ml utilities
import numpy as np

 Connect to GitHub

In [7]:
import subprocess
import os
from google.colab import userdata

GITHUB_TOKEN    = userdata.get("GITHUB_TOKEN")
GITHUB_USERNAME = "the-liyanage"
REPO_NAME       = "neurosynth"

# set git identity — must do this every session
subprocess.run(["git", "config", "--global",
                "user.name", "the-liyanage"])
subprocess.run(["git", "config", "--global",
                "user.email", "hiruniliyanage4@gmail.com"])

# remove the empty folder that's there now
subprocess.run(["rm", "-rf", f"/content/{REPO_NAME}"])
print("Old folder removed")


# clone properly from GitHub — this creates the .git folder
os.chdir("/content")
result = subprocess.run([
    "git", "clone",
    f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
], capture_output=True, text=True)


print(result.stdout)
print(result.stderr)

# verify .git is now there
contents = os.listdir(f"/content/{REPO_NAME}")
print(f"\nContents of /content/{REPO_NAME}:")
print(contents)

if ".git" in contents:
    print(" .git folder found — repo cloned correctly!")
else:
    print(" Still no .git — tell me the error above")

Old folder removed

Cloning into 'neurosynth'...


Contents of /content/neurosynth:
['notebooks', '.git', '.gitignore', 'README.md', 'docker-compose.yml', 'requirements.txt']
 .git folder found — repo cloned correctly!


Download EEG Data

In [8]:
from mne.datasets import eegbci

# where to save the raw downloaded files (on Google Drive)
DATA_DIR = "/content/drive/MyDrive/neurosynth/data/raw"


# People whose brain signals were recoreded (starts with 5)
SUBJECTS = [1, 2, 3, 4, 5]

# Recording sessions (6, 10, 14 are the motor IMAGERY sessions)
RUNS = [6, 10, 14]

print("Downloading EEG Data.... \n")

for subject in SUBJECTS:
  raw_fnames = eegbci.load_data(
      subject,  # which person
      runs = RUNS, # which session
      path = DATA_DIR,   # where to save the data
      verbose = False  # don't print MNE's internal logs

  )

  # raw_fnames is the list of file paths that were downloaded
  print(f"Subject {subject:03d} - {len(raw_fnames)} files downloaded")


Do you want to set the path:
    /content/drive/MyDrive/neurosynth/data/raw
as the default EEGBCI dataset path in the mne-python config [y]/n? y
Subject 001 - 3 files downloaded
Subject 002 - 3 files downloaded
Subject 003 - 3 files downloaded
Subject 004 - 3 files downloaded
Subject 005 - 3 files downloaded


In [9]:
# look inside one EEG file
# concatenate_raws joins multiple recordings into one
from mne.io import concatenate_raws

# load just subject 1, run 6
raw_fnames = eegbci.load_data(
    1,  # subject number
    runs = [6],
    path = DATA_DIR,
    verbose = False
)


# read_raw_edf reads the .edf file (European Data Format)
# .edf is the standard file format for biological signal recordings
raw = mne.io.read_raw_edf(raw_fnames[0], preload = True, verbose = False)


# ch_names is a list of all electrode names
print(f"Channels: {len(raw.ch_names)}")

# sfreq = sampling frequency = how many readings per second
print(f"Sampe rate:  {raw.info["sfreq"]} Hz")

# raw.times is an array of every
print(f"Duration: {raw.times[-1]:.1f} seconds")

# get_data() returns the raw signal as a numpy array
print(f"Data shape: {raw.get_data().shape}")
print(f"            (channels, timepoints)")


print(f"\nFirst 5 channels: {raw.ch_names[:5]}")


# annotations are the labels - timestamps marking when each task happened
# T0 = rest, T1 = left first imagery, T2, = right first imegery
print(f"\nAnnotations")
for ann in raw.annotations:
  print(f"  {ann['onset']:.1f}s "
        f"duration={ann['duration']:.1f}s   " # how long it lasted
        f" label ={ann['description']}' ")  # what task it was



Channels: 64
Sampe rate:  160.0 Hz
Duration: 125.0 seconds
Data shape: (64, 20000)
            (channels, timepoints)

First 5 channels: ['Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.']

Annotations
  0.0s duration=4.2s    label =T0' 
  4.2s duration=4.1s    label =T2' 
  8.3s duration=4.2s    label =T0' 
  12.5s duration=4.1s    label =T1' 
  16.6s duration=4.2s    label =T0' 
  20.8s duration=4.1s    label =T1' 
  24.9s duration=4.2s    label =T0' 
  29.1s duration=4.1s    label =T2' 
  33.2s duration=4.2s    label =T0' 
  37.4s duration=4.1s    label =T1' 
  41.5s duration=4.2s    label =T0' 
  45.7s duration=4.1s    label =T2' 
  49.8s duration=4.2s    label =T0' 
  54.0s duration=4.1s    label =T2' 
  58.1s duration=4.2s    label =T0' 
  62.3s duration=4.1s    label =T1' 
  66.4s duration=4.2s    label =T0' 
  70.6s duration=4.1s    label =T1' 
  74.7s duration=4.2s    label =T0' 
  78.9s duration=4.1s    label =T2' 
  83.0s duration=4.2s    label =T0' 
  87.2s duration=4.1s    label =T2'

In [ ]:
import shutil
from datetime import datetime
import os


REPO_NAME = "neurosynth"

# copy the notebook from Google Drive into our cloned repo folder
# source --> our notebook saved on Google Drive
# destination --> notebooks folder inside the cloned repo

# Ensure the destination directory exists
os.makedirs(f"/content/{REPO_NAME}/notebooks", exist_ok=True)

shutil.copy(
    "/content/drive/MyDrive/Colab Notebooks/01_download_data.ipynb",
    f"/content/{REPO_NAME}/notebooks/01_download_data.ipynb"
)

# move into the repo folder
os.chdir(f"/content/{REPO_NAME}")

subprocess.run(["git", "add", "."])

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
subprocess.run(["git", "commit", "-m",
                "add EEG download notebook"])

result = subprocess.run([
    "git", "push",
    f"https://{userdata.get('GITHUB_TOKEN')}@github.com/the-liyanage/neurosynth.git",
     "main"],
    capture_output=True,
    text=True
    )



print(" Pushed!" if result.returncode == 0 else result.stderr)

 Pushed!


In [ ]:
import os
print("Current folder:", os.getcwd())
print("Contents of /content:")
print(os.listdir("/content"))

Current folder: /content/neurosynth
Contents of /content:
['.config', 'drive', 'neurosynth', 'sample_data']


In [ ]:
contents = os.listdir("/content/neurosynth")
print("Contents of /content/neurosynth:")
print(contents)

Contents of /content/neurosynth:
['.git', 'docker-compose.yml', 'requirements.txt', 'notebooks', '.gitignore', 'README.md']


Preprocessing Settings

In [ ]:
# we define ALL settings in one place
# this is called a configuration block

# ----------Filter settings
# keep only frequencies between 8 and 30 Hz
# below 8 Hz = noise from movement and sweat
# above 30 Hz = muscle noise and power line interference
FREQ_LOW = 8.0 # lower cutoff
FREQ_HIGH = 30.0 # upper cutoff


# -----------Epoch settings
# how long each brain signal window should be
# 0.0 = start exactly at event onset ( when T1 or T2)
# 4.0 = end 4 seconds after onset
# 4 seconds capture the complete motor imagery response
EPOCH_TMIN = 0.0
EPOCH_TMAX = 4.0


# ---------Label settings
# we only want T1 and T2
# T1 = imagine left fist ---> we will convert this to label 0
# T2 = imagine right fist -----> we will onvert this to label 1
EVENT_ID = {"T1": 1, "T2": 2}


OUTPUT_DIR = "/content/drive/MyDrive/neurosynth/data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(" Configuration set")
print(f"   Filter:     {FREQ_LOW} - {FREQ_HIGH} Hz")
print(f"   Epoch:      {EPOCH_TMIN}s to {EPOCH_TMAX}s")
print(f"   Labels:     T1=left fist, T2=right fist")
print(f"   Output:     {OUTPUT_DIR}")



 Configuration set
   Filter:     8.0 - 30.0 Hz
   Epoch:      0.0s to 4.0s
   Labels:     T1=left fist, T2=right fist
   Output:     /content/drive/MyDrive/neurosynth/data/processed


Preprocessing Functions


In [ ]:
# we define functions here — one function per preprocessing step
# functions are reusable blocks of code


def load_subject(subject, runs, data_dir):
    """
    Loads all runs for one subject and joins them
    into one continuous EEG recording.

    Why join runs? Each run is a separate file but
    they're all the same person doing the same task.
    Joining gives us more data per subject.
    """

    # download/load the file paths for this subject
    raw_fnames = eegbci.load_data(
        subject, # Changed from subject=subject
        runs=runs,
        path=data_dir,
        verbose=False
    )

    # read each .edf file into an MNE Raw object
    # a Raw object holds the signal + all metadata
    raws = [
        mne.io.read_raw_edf(f, preload=True, verbose=False)
        for f in raw_fnames
    ]
    # this is called a list comprehension
    # it's a compact way of writing a for loop that builds a list

    # concatenate_raws joins the list of recordings into one
    raw = concatenate_raws(raws)

    # standardize fixes the channel names
    # PhysioNet files use names like "Fc5." with a dot
    # standardize removes the dot → "FC5"
    # this is needed to match the standard 10-20 electrode system
    eegbci.standardize(raw)

    # set electrode positions on the scalp
    # standard_1005 is the international standard electrode layout
    # this tells MNE exactly where each electrode sits on the head
    montage = mne.channels.make_standard_montage("standard_1005")
    raw.set_montage(montage, verbose=False)

    return raw


def apply_bandpass_filter(raw):
    """
    Keeps only frequencies between FREQ_LOW and FREQ_HIGH.
    Everything outside that range is noise for our task.
    """

    raw.filter(
        l_freq=FREQ_LOW,    # l_freq = lower frequency cutoff
        h_freq=FREQ_HIGH,   # h_freq = upper frequency cutoff
        method="iir",       # IIR = Infinite Impulse Response
                            # a type of filter that is fast and efficient
        verbose=False
    )
    return raw


def extract_epochs(raw):
    """
    Finds all T1 and T2 events and cuts the continuous
    signal into fixed 4 second windows around each event.

    Returns:
        X → numpy array shape (n_epochs, 64, 641)
             n_epochs = number of task windows found
             64 = number of EEG channels
             641 = 4 seconds × 160 Hz + 1 timepoints

        y → numpy array shape (n_epochs,)
             0 = left fist imagery
             1 = right fist imagery
    """

    # events_from_annotations converts text labels (T0, T1, T2)
    # into a numpy array of [sample_index, 0, event_id]
    # sample_index tells us exactly which timepoint the event starts
    events, _ = mne.events_from_annotations(raw, verbose=False)
    # _ means we don't need the second return value (event dictionary)

    # Epochs cuts the continuous signal into windows
    # one window per event that matches EVENT_ID
    epochs = mne.Epochs(
        raw,
        events,
        event_id=EVENT_ID,  # only cut windows for T1 and T2
        tmin=EPOCH_TMIN,    # window starts at event onset
        tmax=EPOCH_TMAX,    # window ends 4 seconds later
        baseline=None,      # no baseline correction
                            # we normalize ourselves instead
        preload=True,
        verbose=False
    )

    # get_data() returns the epochs as a numpy array
    # shape = (n_epochs, n_channels, n_timepoints)
    X = epochs.get_data()

    # epochs.events[:, 2] gets the event id column (T1=1 or T2=2)
    # subtracting 1 converts to 0-indexed labels
    # T1=1 → 0 (left fist)
    # T2=2 → 1 (right fist)
    y = epochs.events[:, 2] - 1

    return X, y


def normalize(X):
    """
    Z-score normalization — makes every epoch have mean=0 std=1.

    Why? Raw EEG amplitude varies hugely between subjects
    because of skull thickness, hair, electrode contact quality.
    Normalization removes that variation so the model focuses
    on signal PATTERNS not absolute voltage values.

    X shape: (n_epochs, n_channels, n_timepoints)
    We normalize along the time axis (axis=-1)
    separately for each channel in each epoch.
    """

    # mean of each channel's timeseries per epoch
    # keepdims=True keeps the shape so subtraction works correctly
    mean = X.mean(axis=-1, keepdims=True)

    # standard deviation of each channel's timeseries per epoch
    std  = X.std(axis=-1, keepdims=True)

    # if std is 0 (flat signal) set to 1 to avoid division by zero
    std[std == 0] = 1

    # subtract mean and divide by std
    # result has mean=0 and std=1
    return (X - mean) / std


print("✅ All preprocessing functions defined")

✅ All preprocessing functions defined


In [ ]:
# tqdm gives us a progress bar so we can see how far along we are
from tqdm.notebook import tqdm
import numpy as np

# empty lists to collect data from all subjects
all_X = []   # will hold EEG data arrays
all_y = []   # will hold label arrays

print("Starting preprocessing pipeline...\n")

# loop through every subject
for subject in tqdm(SUBJECTS, desc="Processing subjects"):

    # ── 1. Load ──────────────────────────────────
    raw = load_subject(subject, RUNS, DATA_DIR)

    # ── 2. Filter ────────────────────────────────
    raw = apply_bandpass_filter(raw)

    # ── 3. Epoch ─────────────────────────────────
    X, y = extract_epochs(raw)

    # skip this subject if too few valid epochs found
    # can happen if the recording had issues
    if len(X) < 5:
        print(f"⚠️  Subject {subject} skipped — too few epochs")
        continue
    # continue means skip the rest of this loop iteration
    # and move to the next subject

    # ── 4. Normalize ─────────────────────────────
    X = normalize(X)

    # add this subject's data to our collection
    all_X.append(X)
    all_y.append(y)

    print(f"  ✅ Subject {subject:03d} — "
          f"{X.shape[0]} epochs | shape {X.shape}")

# np.concatenate joins the list of arrays into one big array
# axis=0 means stack along the first dimension (epochs)
all_X = np.concatenate(all_X, axis=0)
all_y = np.concatenate(all_y, axis=0)

print(f"\n✅ Pipeline complete!")
print(f"   Total epochs:  {all_X.shape[0]}")
print(f"   X shape:       {all_X.shape}")
print(f"                   ↑ (epochs, channels, timepoints)")
print(f"   y shape:       {all_y.shape}")
print(f"   Left fist:     {(all_y==0).sum()} epochs")
print(f"   Right fist:    {(all_y==1).sum()} epochs")

Starting preprocessing pipeline...



Processing subjects:   0%|          | 0/5 [00:00<?, ?it/s]

  ✅ Subject 001 — 66 epochs | shape (66, 64, 641)
  ✅ Subject 002 — 69 epochs | shape (69, 64, 641)
  ✅ Subject 003 — 66 epochs | shape (66, 64, 641)
  ✅ Subject 004 — 67 epochs | shape (67, 64, 641)
  ✅ Subject 005 — 68 epochs | shape (68, 64, 641)

✅ Pipeline complete!
   Total epochs:  336
   X shape:       (336, 64, 641)
                   ↑ (epochs, channels, timepoints)
   y shape:       (336,)
   Left fist:     225 epochs
   Right fist:    111 epochs


In [ ]:
# np.save saves a numpy array to disk as a .npy file


np.save(f"{OUTPUT_DIR}/X.npy", all_X)
#saves the EEG data array

np.save(f"{OUTPUT_DIR}/y.npy", all_y)
#saves the labels array

print(f"  {OUTPUT_DIR}/X.npy")
print(f"  {OUTPUT_DIR}/y.npy")

  /content/drive/MyDrive/neurosynth/data/processed/X.npy
  /content/drive/MyDrive/neurosynth/data/processed/y.npy


In [ ]:
X_check = np.load(f"{OUTPUT_DIR}/X.npy")
y_check = np.load(f"{OUTPUT_DIR}/y.npy")

print(f"X shape:       {X_check.shape}")
print(f"y shape:       {y_check.shape}")
print(f"Data min:      {X_check.min():.3f}")
print(f"Data max:      {X_check.max():.3f}")
print(f"Data mean:     {X_check.mean():.6f}")     # should be near 0
print(f"Data  std:     {X_check.std():.6f}")      # should be near 1
print(f"Unique labels: {np.unique(y_check)}")

X shape:       (336, 64, 641)
y shape:       (336,)
Data min:      -10.667
Data max:      9.188
Data mean:     0.000000
Data  std:     1.000000
Unique labels: [0 1]


In [ ]:
import shutil
from datetime import datetime

# copy notebook from Drive into the cloned repo
shutil.copy(
    "/content/drive/MyDrive/neurosynth/01_download_data.ipynb",
    f"/content/neurosynth/notebooks/01_download_data.ipynb"
)

# go into repo
os.chdir("/content/neurosynth")

# stage everything
subprocess.run(["git", "add", "."])

# commit with timestamp
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
subprocess.run(["git", "commit", "-m", f"update: {timestamp}"])

# push
result = subprocess.run(
    ["git", "push",
     f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git",
     "main"],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ Pushed to GitHub!")
else:
    print(result.stderr)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/neurosynth/01_download_data.ipynb'